# spdconv_bifpn_p2 **v2** — dataset-2 (`ke-project/yolo-tb1l6`)

## POST-MORTEM: ilk v2 kosusu neden 0.5718 geldi

| kosu | mAP50 | mAP50-95 | ultralytics | optimizer / lr0 | epoch |
|---|---|---|---|---|---|
| `spdconv_bifpn-2` (baseline) | 0.9105 | **0.6666** | 8.4.118 | SGD 0.01 | 499 |
| `spdconv_bifpn` (baseline) | 0.9074 | **0.6660** | 8.4.118 | SGD 0.01 | 500 |
| **`v2_s_bifpn3in_caa`** | 0.8458 | **0.5718** | **8.4.14** | **AdamW 0.001** | **415** |

### Duzeltme: veri seti AYNI
Onceki notumda eski kosularin farkli veri setinde oldugunu yazmistim, **yanlisti**.
`../../../YOLO.v1i.yolo26` tasinmis; etiketler byte-byte ayni (md5 dogrulandi:
train `2a74bc7f7880`, valid `cb5247c03a5a`, test `38efd00e3cb1`).
Yani **0.6660 gecerli bir taban** ve v2 onun 0.094 altinda kaldi.

### Ana sebep: optimizer/LR
Dusuk lr'nin bu veri setinde cokturdugu **senin kendi kosularinda** zaten kanitli
(ayni YAML, ayni optimizer, sadece lr0 degisiyor):

| model | lr0 | mAP50-95 |
|---|---|---|
| `spdconv_bifpn.yaml` + SGD | **0.01** | **0.6215 / 0.6235 / 0.6251** |
| | 0.0003 | 0.5346 / 0.5403 |
| | 0.0002 | 0.4983 |
| `yolo26n_repc3k2_bifpn_consam.yaml` + SGD | **0.01** | **0.6123** |
| | 0.001 | 0.4557 / 0.3793 |

lr0'i 0.01'den dusurmek **0.16–0.23 mAP50-95** goturuyor. AdamW 0.001 onerisi bu
kaniti gormeden verildi.

**Ama dikkat:** v2 mimarisi `lr0=0.001`'de **0.5718** aldi; senin ayni lr'deki en
iyi kosun **0.4557**'ydi. Mimari muhtemelen calisiyor, recete sabote etti.

### Ikinci sebep: ultralytics surumu
Butun eski kosular **8.4.118**, v2 kosusu **8.4.14**. Eski `args.yaml`'larda
`dis / dlam / dlog / dgrad / cls_remap / cls_pw / channels_last` var, yenide yok.
(`distill_model: null` idi, yani distilasyon acik degildi — fark oradan gelmiyor;
ama 104 patch surum arasi var ve **kosular karsilastirilabilir degil**.)

### Ucuncu sebep: ayni anda 6 degisken degisti
optimizer, lr0, ultralytics surumu, `cos_lr`, `warmup`, `flipud`, `scale`,
`translate`, `close_mosaic` **ve** mimari. Boyle bir kosudan hicbir sey ogrenilemez.

### Dorduncu / besinci
Kosu 415/500'de kesildi (mAP hala tirmaniyordu: 400 → 0.5705, 415 → 0.5718) ve
`s` degil **`n`** yaml'i kosuldu.

---

## Bu notebook artik ne yapiyor
Baseline'a gore **sadece mimariyi** degistiriyor. Hiperparametreler
`ablasion-all/runs/detect/spdconv_bifpn/args.yaml` ile birebir ayni.

### 0) Surum eslestirme (karsilastirma yapacaksan sart)

```bash
pip install ultralytics==8.4.118
```

> Bu, elle eklenmis `ultralytics/nn/modules/inn_modules_v2.py` yamasini **siler**
> (`SPDConv, CoTAttention, AAM, FEM, RepCSP, AFF_Add2` + `nn/modules/__init__.py`
> ve `nn/tasks.py` import satirlari). Kurulumdan sonra geri koy;
> `nn/modules/__init__.py.bak_inn_yolo26_v2` yedegi duruyor.
>
> `modules_v2.py` bu yamaya **bagimli degil** — host modul olarak upstream'de
> garanti olan `Focus` / `C3Ghost` kullaniyor. Ama eski YAML'larin (`AAM`, `FEM`,
> `RepCSP`, `AFF_Add2` kullananlar) yamaya ihtiyaci var.

Asagidaki hucre surumu kontrol eder.

In [1]:
import os, sys, torch, ultralytics
# Bu notebook spdconv-bifpn-p2/ klasorunden calistirilmali (relatif yollar buna gore).

ABLASION = os.path.abspath("../ablasion-all")
sys.path.insert(0, ABLASION)

DATA = "../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml"   # ke-project/yolo-tb1l6
device = 0 if torch.cuda.is_available() else "cpu"

print("ultralytics:", ultralytics.__version__)
if ultralytics.__version__ != "8.4.118":
    print("  !! Eski kosularin hepsi 8.4.118 ile yapildi.")
    print("  !! Bu kosu onlarla KARSILASTIRILABILIR DEGIL.")
print("cihaz      :", "GPU cuda:0" if device == 0 else "CPU")
print("data       :", os.path.abspath(DATA), "->", os.path.exists(DATA))

ultralytics: 8.4.14
  !! Eski kosularin hepsi 8.4.118 ile yapildi.
  !! Bu kosu onlarla KARSILASTIRILABILIR DEGIL.
cihaz      : GPU cuda:0
data       : c:\Users\ertug\Desktop\yolo\dataset2\yolo26\YOLO.v1i.yolo26\data.yaml -> True


### 1) Custom modul kaydi

`register_v2()` **`YOLO(...)` satirindan once** cagrilmali:

* `Concat` → `WeightedConcatN` (BiFPN, N girisli, olcek-koruyan)
* `parse_model` wrapper'i `CAA` / `C3k2CAA` isimlerini calisma aninda secilen
  host isimlere (`Focus` / `C3Ghost`) cevirir — site-packages'a dokunulmaz,
  lazy build yok.

`patch_concat_to_hybrid()` **cagrilmiyor** (v1'deki cakismanin sebebi oydu).
WIoU de kapali (`train/box_loss` 1. epoch'ta 295–344'e ciktigi icin).

In [2]:
import ultralytics.nn.tasks as tasks
from spdconv import SPDConv
tasks.SPDConv = SPDConv

from modules_v2 import register_v2
register_v2(bifpn=True)          # bifpn=False -> ablasyon: duz Concat

from ultralytics import YOLO

[modules_v2] kayit tamam (ultralytics 8.4.14):
   CAA      -> Context Anchor Attention  [host: Focus]
   C3k2CAA  -> C3k2 + CAA                [host: C3Ghost]
   Concat   -> WeightedConcatN (BiFPN, N girisli, olcek-koruyan)


### 2) Modeli kur + kendi SPD-Conv / BiFPN agirliklarini yukle

COCO yerine bu veri setinde **zaten egitilmis kendi checkpoint'lerin**
kullaniliyor. Olculen kapsama (`intersect_dicts`, isim+shape eslesmesi):

| kaynak | tensor | parametre |
|---|---|---|
| COCO `yolo26n.pt` | 353/1118 | %20.2 |
| `runs/detect/train-2` (duz YOLO26n) | 353/1118 | %20.2 |
| `BIFPN/ablation_bifpn_v1` (yalniz BiFPN) | 353/1118 | %20.2 |
| `SPD-Conv/ablation_spd_conv_v` (yalniz SPD-Conv) | 358/1118 | **%58.7** |

Ucurumun sebebi backbone'daki SPD-Conv katmanlari (`model.0/1/3/5/7`):
SPDConv'un conv agirligi `c2 x (4*c1) x k x k`, stok `Conv` ise
`c2 x c1 x k x k`. COCO ya da SPD-Conv'suz bir checkpoint o katmanlarin
sadece BatchNorm'unu doldurabiliyor - ve bunlar modelin en agir katmanlari.

**Zincirde SADECE tek tek bilesen ablasyonlari var:**
`train-2` (duz YOLO26n) -> `bifpn` (yalniz BiFPN) -> `spdconv` (yalniz SPD-Conv).

> Birlesik `ablasion-all/spdconv_bifpn{,-2}` checkpoint'leri **bilerek disarida**.
> Onlar v2'nin yenmeye calistigi baseline'in ta kendisi (0.6660 / 0.6666);
> onlardan baslatmak, olcmeye calistigimiz birlesimin 500 epoch'luk
> cozumunden baslamak olurdu ve "baseline'i gectik" iddiasini gecersiz kilardi.

`load_chain()` zinciri sirayla yukler (son cagri kazanir) ve her adimda kac
tensor'un YENI geldigini, kacinin uzerine yazildigini raporlar. Raporda
gorulecegi gibi `train-2` ve `bifpn` sifir yeni tensor katiyor - ikisinin
doldurdugu her anahtari `spdconv` da dolduruyor ve uzerine yaziyor. Yani sonuc
`--pretrained spdconv` ile birebir ayni; zincir mevcut `ablasion-main.ipynb`
akisinla tutarli olsun ve bu durum GORUNUR olsun diye duruyor.

> Eski SPD-Conv checkpoint'leri `ultralytics.nn.modules.spdconv` modulunu
> ariyor (o zamanki dosya yolu, artik yok) ve duz `torch.load` ile
> `No module named ...` hatasi veriyor. `pretrained_v2.install_legacy_shims()`
> bu ismi projedeki `spdconv.SPDConv`'a bagliyor; `load_chain()` bunu
> otomatik cagiriyor.

Baseline `spdconv_bifpn` kosusu `n` scale'di, dolayisiyla **once `n`** ile
karsilastir. `s` (19.6M) ondan sonra.

In [3]:
from pretrained_v2 import load_chain, report

SCALE = "n"                       # once "n" (baseline ile karsilastirilabilir), sonra "s"
CFG = os.path.join(ABLASION, f"yolo26{SCALE}-spdconv-bifpn-p2-v2.yaml")

model = YOLO(CFG)

# her kaynagin tek basina ne kadar ortustugu
print("=== kaynak bazinda kapsama ===")
report(model)
print()

# zincir: zayiftan iyiye, son cagri kazanir
model = load_chain(model, ["train-2", "bifpn", "spdconv"])

print("\nparametre: %.2fM" % (sum(p.numel() for p in model.model.parameters()) / 1e6))

=== kaynak bazinda kapsama ===
train-2                     353/1118 tensor   20.2% parametre
bifpn                       353/1118 tensor   20.2% parametre
spdconv                     358/1118 tensor   58.7% parametre

[pretrained] hedef: 1118 tensor / 5.03M parametre
kaynak                       eslesen   YENI  uzerine yazilan  kumulatif param
------------------------------------------------------------------------------
Transferred 354/1118 items from pretrained weights
train-2                          353    353                0            20.2%
Transferred 354/1118 items from pretrained weights
bifpn                            353      0              353            20.2%
Transferred 358/1118 items from pretrained weights
spdconv                          358      5              353            58.7%
------------------------------------------------------------------------------
toplam dolan: 358/1118 tensor (58.7% parametre), rastgele kalan: 760 tensor
  (rastgele kalanlar: CAA katmanl

### 3) Egitim — KANITLANMIS recete

Asagidaki hiperparametreler `runs/detect/spdconv_bifpn/args.yaml` ile **birebir
ayni**. Tek fark mimari. Ilk v2 kosusundaki AdamW/`cos_lr`/`flipud`/`scale`/
`close_mosaic` degisikliklerinin **hicbiri** burada yok — once temiz bir
karsilastirma, sonra tek tek deneme.

In [4]:
results = model.train(
    data=DATA,
    epochs=500,
    imgsz=640,
    batch=16,           # baseline ile ayni. 's' scale'de 8GB'a sigmaz -> 8 yap.
    device=device,
    workers=4,
    name=f"v2_{SCALE}_proven",
    patience=100,

    # --- spdconv_bifpn (0.6660) ile BIREBIR AYNI ---
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    cos_lr=False,
    warmup_epochs=3.0,
    momentum=0.937,
    weight_decay=0.0005,
    nbs=64,
    box=7.5, cls=0.5, dfl=1.5,
    scale=0.5,
    translate=0.1,
    mosaic=1.0,
    close_mosaic=10,
    fliplr=0.5,
    flipud=0.0,
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    mixup=0.0, cutmix=0.0,

    plots=True,
    val=True,
)

New https://pypi.org/project/ultralytics/8.4.133 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.12.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=c:\Users\ertug\Desktop\yolo\codes\yolov26-dat

### 4) Saglik kontrolu — ILK EPOCH'TAN SONRA BAK

`train/box_loss` **2–5** araliginda olmali. 20'nin ustundeyse kayip olcegi
patlamis (WIoU hatasinin imzasi) — kosuyu durdur.

Ayrica **kosuyu yarida kesme.** Ilk v2 kosusu 415/500'de kesildi ve mAP hala
tirmaniyordu.

In [6]:
import csv, glob
run = sorted(glob.glob(f"runs/detect/v2_{SCALE}_proven*/results.csv"))[-1]
rows = list(csv.DictReader(open(run)))
print(run, "| epoch:", len(rows))
for r in rows[:2] + rows[-3:]:
    print("ep %-4s box=%-8s cls=%-8s mAP50=%-7s mAP50-95=%s" % (
        r["epoch"], r["train/box_loss"], r["train/cls_loss"],
        r["metrics/mAP50(B)"], r["metrics/mAP50-95(B)"]))

b = max(rows, key=lambda r: float(r["metrics/mAP50-95(B)"]))
print("\nEN IYI ep %s -> P %.3f R %.3f mAP50 %.4f mAP50-95 %.4f" % (
    b["epoch"], float(b["metrics/precision(B)"]), float(b["metrics/recall(B)"]),
    float(b["metrics/mAP50(B)"]), float(b["metrics/mAP50-95(B)"])))
print("TABAN (spdconv_bifpn)          mAP50 0.9074 mAP50-95 0.6660")

box1 = float(rows[0]["train/box_loss"])
print("\n[saglik] 1. epoch train/box_loss = %.2f -> %s" % (
    box1, "OK" if box1 < 20 else "!! KAYIP OLCEGI PATLAMIS, DURDUR"))
if len(rows) < 500:
    print("[uyari] kosu %d epoch'ta bitmis, 500 degil - yarida kesilmis olabilir" % len(rows))

runs/detect\v2_n_proven\results.csv | epoch: 500
ep 1    box=3.52145  cls=7.92565  mAP50=0.0003  mAP50-95=7e-05
ep 2    box=3.35581  cls=5.27301  mAP50=0.0096  mAP50-95=0.00288
ep 498  box=0.83393  cls=0.39398  mAP50=0.84546 mAP50-95=0.57655
ep 499  box=0.85806  cls=0.40219  mAP50=0.84523 mAP50-95=0.57638
ep 500  box=0.82425  cls=0.39101  mAP50=0.84537 mAP50-95=0.57631

EN IYI ep 440 -> P 0.852 R 0.784 mAP50 0.8513 mAP50-95 0.5883
TABAN (spdconv_bifpn)          mAP50 0.9074 mAP50-95 0.6660

[saglik] 1. epoch train/box_loss = 3.52 -> OK


### 5) Test seti

In [7]:
from pathlib import Path
best = Path(model.trainer.best)
print("test ediliyor:", best)

res = YOLO(str(best)).val(data=DATA, split="test", imgsz=640, device=device)
d = res.results_dict
print("\n=== TEST ===")
for k, lbl in [("metrics/precision(B)", "Precision"),
               ("metrics/recall(B)", "Recall"),
               ("metrics/mAP50(B)", "mAP@0.5"),
               ("metrics/mAP50-95(B)", "mAP@0.5:0.95")]:
    v = d.get(k)
    print(f"{lbl:>14}: {v*100:.2f}%" if v is not None else f"{lbl:>14}: yok")

try:
    for i, c in enumerate(res.names.values()):
        print(f"  {c:>6}: mAP50={res.box.ap50[i]:.4f}  mAP50-95={res.box.ap[i]:.4f}")
except Exception as e:
    print("sinif bazinda metrik alinamadi:", e)

test ediliyor: C:\Users\ertug\Desktop\yolo\codes\yolov26-dataset-2\spdconv-bifpn-p2\runs\detect\v2_n_proven3\weights\best.pt
Ultralytics 8.4.14  Python-3.12.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
YOLO26n-spdconv-bifpn-p2-v2 summary: 229 layers, 4,768,504 parameters, 0 gradients
val: Fast image access  (ping: 0.00.0 ms, read: 156.243.6 MB/s, size: 26.7 KB)
val: Scanning C:\Users\ertug\Desktop\yolo\dataset2\yolo26\YOLO.v1i.yolo26\test\labels.cache... 279 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 279/279  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 325, len(boxes) = 1023. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 6.0it/s 3.0s0.2s
                   all        279       1023      0.9

### 6) Ablasyon sirasi

**Her seferinde TEK degisken.** `train_v2.py` ayni receteyi terminalden verir:

```bash
python train_v2.py --scale n                    # 1. mimari etkisi (baseline ile kiyasla)
python train_v2.py --scale s                    # 2. scale etkisi
python train_v2.py --scale n --no-bifpn         # 3. BiFPN etkisi
python train_v2.py --scale n --set flipud=0.5   # 4. tek augmentasyon denemesi
python train_v2.py --scale n --set scale=0.6
python train_v2.py --recipe experimental        # ilk v2 kosusunun ayarlari (referans)
```

Taban: `spdconv_bifpn` → mAP50 **0.9074** / mAP50-95 **0.6660**
(valid, SGD 0.01, ultralytics 8.4.118).